# Predicting Stellar Class — LightGBM Solution

This notebook trains a LightGBM classifier with engineered features and target encoding for the 3-class stellar classification task (GALAXY, QSO, STAR). Evaluation metric: **balanced accuracy**.

In [190]:
import os

import numpy as np
import pandas as pd

import lightgbm as lgb
from lightgbm import LGBMClassifier

## Configuration

In [191]:
ID_COL = 'id'
TARGET_COL = 'class'
CATEGORICAL_COLS = ['spectral_type', 'galaxy_population']

DATA_DIR = '/kaggle/input/competitions/playground-series-s6e6'
if not os.path.isdir(DATA_DIR):
    DATA_DIR = 'data'

## Load Data

Competition files: `train.csv`, `test.csv`, `sample_submission.csv` at `/kaggle/input/competitions/playground-series-s6e6/`. Locally, fallback to `data/`.

In [192]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f'Train: {train_df.shape}, Test: {test_df.shape}')

Train: (577347, 12), Test: (247435, 11)


## Feature Engineering

Derived features:
- Color indices (u-g, g-r, r-i, r-z)
- 3D unit-sphere coordinates (`coord_x`, `coord_y`, `coord_z`) and sin/cos transforms for alpha and delta
- Coordinate distance and bins
- Absolute magnitude in r-band (`abs_mag_r`)
- Log-redshift (`redshift_log`) — raw redshift is dropped after engineering

In [193]:
from sklearn.preprocessing import KBinsDiscretizer

def get_absolute_magnitude(m_apparent, z):
    z = np.asarray(z, dtype=float)
    m_apparent = np.asarray(m_apparent, dtype=float)
    mu = np.zeros_like(z, dtype=float)
    mask = z > 1e-4
    if mask.any():
        z_valid = z[mask]
        d_L = (299792.458 / 70) * z_valid * (1 + 0.775 * z_valid)
        mu[mask] = 5 * np.log10(d_L * 1e6) - 5
    return m_apparent - mu

def add_features(df):
    """
    Добавляет физические и геометрические признаки к датасету.
    
    Включает:
    - Цветовые индексы (color indices)
    - Тригонометрическое кодирование координат
    - 3D координаты на единичной сфере
    - Абсолютную звёздную величину (abs_mag_r)
    - Логарифм redshift
    - Взаимодействия признаков
    - SSFR proxy (индикатор звездообразования)
    """
    d = df.copy()
    
    # ЦВЕТОВЫЕ ИНДЕКСЫ (Color indices)
    d['color_ug'] = d['u'] - d['g']
    d['color_gr'] = d['g'] - d['r']
    d['color_ri'] = d['r'] - d['i']
    d['color_rz'] = d['r'] - d['z']
    
    # РАССТОЯНИЕ ОТ НАЧАЛА КООРДИНАТ
    d['coord_dist'] = np.sqrt(d['alpha']**2 + d['delta']**2)
    
    # ТРИГОНОМЕТРИЧЕСКОЕ КОДИРОВАНИЕ КООРДИНАТ
    d['sin_alpha'] = np.sin(np.radians(d['alpha']))
    d['cos_alpha'] = np.cos(np.radians(d['alpha']))
    d['sin_delta'] = np.sin(np.radians(d['delta']))
    d['cos_delta'] = np.cos(np.radians(d['delta']))
    
    # 3D КООРДИНАТЫ НА ЕДИНИЧНОЙ СФЕРЕ
    d['coord_x'] = np.cos(np.radians(d['delta'])) * np.cos(np.radians(d['alpha']))
    d['coord_y'] = np.cos(np.radians(d['delta'])) * np.sin(np.radians(d['alpha']))
    d['coord_z'] = np.sin(np.radians(d['delta']))
    
    # БИНИНГ КООРДИНАТ
    d['alpha_bin'] = pd.cut(d['alpha'], bins=10, labels=False)
    d['delta_bin'] = pd.cut(d['delta'], bins=10, labels=False)
    
    # АБСОЛЮТНАЯ ЗВЁЗДНАЯ ВЕЛИЧИНА (Hubble law)    
    d['abs_mag_r'] = get_absolute_magnitude(d['r'], d['redshift'])

    # ЛОГАРИФМ REDSHIFT (удаляем сырой redshift)
    d['redshift_log'] = np.log1p(d['redshift'])
    d = d.drop(columns=['redshift'])
    
    # ВЗАИМОДЕЙСТВИЕ: coordinate distance × log-redshift
    d['coord_x_redshift'] = d['coord_dist'] * d['redshift_log']

    return d

train_df = add_features(train_df)
test_df = add_features(test_df)

print(f"Train shape after feature engineering: {train_df.shape}")
print(f"Test shape after feature engineering: {test_df.shape}")
print(f"\nNew features: {[c for c in train_df.columns if c not in ['id', 'class', 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'spectral_type', 'galaxy_population']]}")

Train shape after feature engineering: (577347, 28)
Test shape after feature engineering: (247435, 27)

New features: ['color_ug', 'color_gr', 'color_ri', 'color_rz', 'coord_dist', 'sin_alpha', 'cos_alpha', 'sin_delta', 'cos_delta', 'coord_x', 'coord_y', 'coord_z', 'alpha_bin', 'delta_bin', 'abs_mag_r', 'redshift_log', 'coord_x_redshift']


## Target Encoding

Categorical columns are replaced with smoothed class probabilities per category (one column per target class).

In [194]:
def target_encode_multi(train_df, test_df, col, target_col, smoothing=10):
    # Adds columns {col}_{class}_prob for each target class
    classes = sorted(train_df[target_col].unique())
    global_probs = train_df[target_col].value_counts(normalize=True)
    agg = train_df.groupby(col)[target_col].value_counts(normalize=True).unstack(fill_value=0)
    counts = train_df.groupby(col)[target_col].count()

    for cls in classes:
        if cls not in agg.columns:
            agg[cls] = 0.0
        agg[cls] = (agg[cls] * counts + smoothing * global_probs.get(cls, 0)) / (counts + smoothing)
        train_df[f'{col}_{cls}_prob'] = train_df[col].map(agg[cls])
        test_df[f'{col}_{cls}_prob'] = test_df[col].map(agg[cls])

    return train_df, test_df


train_df, test_df = target_encode_multi(train_df, test_df, 'spectral_type', TARGET_COL, smoothing=20)
train_df, test_df = target_encode_multi(train_df, test_df, 'galaxy_population', TARGET_COL, smoothing=20)

train_df = train_df.drop(columns=CATEGORICAL_COLS)
test_df = test_df.drop(columns=CATEGORICAL_COLS)

## Prepare Matrices

In [195]:
X_train = train_df.drop(columns=[ID_COL, TARGET_COL])
y_train = train_df[TARGET_COL]
X_test = test_df.drop(columns=[ID_COL])
test_ids = test_df[ID_COL]

## Model Hyperparameters

In [ ]:
FINAL_PARAMS = {
    'n_estimators': 3000,
    'learning_rate': 0.01,
    'max_depth': 12,
    'num_leaves': 127,
    'min_child_samples': 5,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'min_gain_to_split': 0.01,
    'class_weight': 'balanced',
    'random_state': 42,
    'verbose': -1,
}

## Train Model

Train LightGBM on the full training set and predict test labels.

In [197]:
model = LGBMClassifier(**FINAL_PARAMS)
model.fit(
    X_train, y_train,
    callbacks=[lgb.log_evaluation(period=500)],
)

predictions = model.predict(X_test)
print(f'Predictions: {len(predictions)}')
print(pd.Series(predictions).value_counts())

Predictions: 247435
GALAXY    158163
QSO        51015
STAR       38257
Name: count, dtype: int64


## Submission

In [198]:
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET_COL: predictions.flatten()
})

submission.to_csv('submission.csv', index=False)